In [1]:

import sqlite3
from typing import Optional

from langchain_core.messages import trim_messages
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode

/home/bc/projects/autonomous_coding_tool/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# We'll extend MessagesState to include a user_id for long-term memory and a code_context for the current file.

class CodingAgentState(MessagesState):
    user_id: Optional[str] = None  # now optional with default
    current_file: Optional[str] = None
    code_context: str = ""
    needs_summary: bool = False

In [3]:
# Smallest qwen model with 2k context
llm = ChatOllama(
    model="qwen3.5:2b",
    temperature=0,
    num_ctx=2048,  # Small context window
    # TO DO: experiment with reduced batch size, etc.
)

In [4]:
# Tools

@tool
def read_file(path: str) -> str:
    """Read the content of a file."""
    try:
        with open(path, 'r') as f:
            return f.read()
    except Exception as e:
        return f"Error: {e}"


@tool
def write_file(path: str, content: str) -> str:
    """Write content to a file."""
    try:
        with open(path, 'w') as f:
            f.write(content)
        return f"Successfully wrote to {path}"
    except Exception as e:
        return f"Error: {e}"


tools = [read_file, write_file]
tool_node = ToolNode(tools)

In [5]:
# Message Trimmer
# We'll use trim_messages to keep only the last N messages, but we can also include a system prompt and the most recent interactions.

from langchain_core.messages import BaseMessage


def count_tokens_for_messages(messages: list[BaseMessage]) -> int:
    """Count total tokens for a list of messages."""
    total = 0
    for msg in messages:
        # Handle both string content and list content (if any)
        content = msg.content
        if isinstance(content, str):
            total += llm.get_num_tokens(content)
        elif isinstance(content, list):
            # If content is a list of dicts (e.g., multimodal), convert to string
            total += llm.get_num_tokens(str(content))
    return total


def trim_history(state: CodingAgentState) -> list:
    """Trim message history to fit within context window."""
    trimmed = trim_messages(
        state["messages"],
        max_tokens=1500,
        strategy="last",
        token_counter=count_tokens_for_messages,  # ✅ Use the wrapper
        include_system=True,
        allow_partial=False,
        start_on="human",  # Optional: start trimming after a human message
        end_on=("human", "tool"),  # Optional: end on human or tool message
    )
    return trimmed

In [6]:
import pickle
from pathlib import Path
from typing import Dict, Any, Optional, List
from langgraph.store.memory import InMemoryStore
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import HumanMessage, SystemMessage


# ----------------------------
# Persistent Memory Store (unchanged)
# ----------------------------
class PersistentMemoryStore:
    def __init__(self, file_path: str = "longterm_memory.pkl"):
        self.file_path = Path(file_path)
        self.store = InMemoryStore()
        self._load()

    def _load(self):
        if self.file_path.exists():
            try:
                with open(self.file_path, "rb") as f:
                    data = pickle.load(f)
                for namespace, memories in data.items():
                    for key, value in memories.items():
                        self.store.put(namespace, key, value)
                print(f"[Memory] Loaded {len(data)} namespaces")
            except Exception as e:
                print(f"[Memory] Error loading: {e}")
        else:
            print("[Memory] No existing memory file, starting fresh.")

    def _save(self):
        try:
            data = self.store._data
            with open(self.file_path, "wb") as f:
                pickle.dump(data, f)
        except Exception as e:
            print(f"[Memory] Error saving: {e}")

    def put(self, namespace: tuple, key: str, value: Dict[str, Any]):
        self.store.put(namespace, key, value)
        self._save()

    def get(self, namespace: tuple, key: str) -> Optional[Dict[str, Any]]:
        return self.store.get(namespace, key)

    def search(self, namespace: tuple, *, limit: int = 10) -> List[Any]:
        items = list(self.store._data.get(namespace, {}).values())
        return items[-limit:]

    def delete(self, namespace: tuple, key: str):
        self.store.delete(namespace, key)
        self._save()


# Global instance
_memory_store = None


def get_memory_store() -> PersistentMemoryStore:
    global _memory_store
    if _memory_store is None:
        _memory_store = PersistentMemoryStore()
    return _memory_store


# ----------------------------
# LangGraph Node Functions
# ----------------------------
def retrieve_memories(state: CodingAgentState, config: RunnableConfig) -> Dict[str, Any]:
    """Retrieve long‑term memories and inject them as a system message."""
    user_id = state.get("user_id", "default_user")
    namespace = ("user_memories", user_id)

    store = get_memory_store()
    memories = store.search(namespace, limit=3)
    print(f"[DEBUG] retrieve_memories: found {len(memories)} memories for user {user_id}")

    if not memories:
        return {}

    # Each memory is an Item object with a .value dict
    memory_text = "\n".join([m.value.get("content", "") for m in memories])
    memory_prompt = SystemMessage(
        content=f"[User preferences/patterns]\n{memory_text}\n\nUse this context when responding. For example, if the user prefers type hints, always include them."
    )
    return {"messages": [memory_prompt] + state.get("messages", [])}


def store_memory(state: CodingAgentState, config: RunnableConfig) -> Dict[str, Any]:
    """Store a memory based on the user's last message (hardcoded for testing)."""
    user_id = state.get("user_id", "default_user")
    namespace = ("user_memories", user_id)

    # Find the last user message
    last_user_msg = None
    for m in reversed(state.get("messages", [])):
        if isinstance(m, HumanMessage):
            last_user_msg = m.content
            break

    if not last_user_msg:
        return {}

    # Simple keyword detection – store a clean memory
    if "type hints" in last_user_msg.lower():
        key = "pref_type_hints"
        memory_content = "User prefers type hints in Python functions."
        store = get_memory_store()
        store.put(namespace, key, {"content": memory_content})
        print(f"[Memory] Stored: {memory_content}")
    else:
        print("[Memory] No relevant preference detected, skipping store.")

    return {}

In [13]:
from langchain_core.messages import AIMessage




In [14]:
# Define the agent node
def agent(state: CodingAgentState) -> Dict[str, Any]:
    """Call the LLM with trimmed history and optional memory."""
    trimmed = trim_history(state)
    system_msg = SystemMessage(
        content=(
            "You are an autonomous coding assistant with access to tools: write_file(path, content).\n"
            "When asked to save code to a file, you MUST call the write_file tool. DO NOT write Python code that opens files.\n"
            "Example:\n"
            "User: 'Save a function to adder.py'\n"
            "Assistant: (calls write_file(path='adder.py', content='def add(a, b): return a + b'))\n"
            "Never output raw code blocks with open() calls. Always use the tool."
        )
    )
    messages = [system_msg] + trimmed
    print(f"[DEBUG] agent received {len(messages)} messages (after trimming)")
    response = llm.invoke(messages)
    return {"messages": [response]}

def auto_save_code(state: CodingAgentState) -> Dict[str, Any]:
    """If the assistant output a code block but no tool call, save it manually."""
    last_msg = state["messages"][-1]
    if not isinstance(last_msg, AIMessage):
        return {}
    content = last_msg.content
    # Check if there's a code block
    import re
    match = re.search(r"```python\n(.*?)\n```", content, re.DOTALL)
    if not match:
        return {}
    code = match.group(1)
    # Try to extract filename from the user's last message
    user_msgs = [m for m in state["messages"] if isinstance(m, HumanMessage)]
    if not user_msgs:
        return {}
    user_last = user_msgs[-1].content
    file_match = re.search(r"(\w+\.py)", user_last)
    filename = file_match.group(1) if file_match else "generated.py"
    # Call the write_file tool directly
    from langchain_core.tools import tool
    write_file.invoke({"path": filename, "content": code})
    print(f"[Auto-save] Wrote code to {filename}")
    # Add a confirmation message
    confirm = SystemMessage(content=f"Code automatically saved to {filename}")
    return {"messages": state["messages"] + [confirm]}

# Build the graph
builder = StateGraph(CodingAgentState)

# Add nodes (no summarize node)
builder.add_node("agent", agent)
builder.add_node("tools", tool_node)
builder.add_node("retrieve_memory", retrieve_memories)
builder.add_node("store_memory", store_memory)
builder.add_node("auto_save_code", auto_save_code)

# Edges
builder.add_edge(START, "retrieve_memory")
builder.add_edge("retrieve_memory", "agent")

builder.add_edge("agent", "auto_save_code")

def should_continue(state: CodingAgentState) -> str:
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    # No summarization – always go to store_memory
    return "store_memory"

builder.add_conditional_edges("agent", should_continue, {
    "tools": "tools",
    "store_memory": "store_memory",
})

builder.add_edge("tools", "store_memory")
builder.add_edge("store_memory", END)  # ✅ end after storing

# Compile with checkpointer and persistent store
conn = sqlite3.connect("checkpoints.db", check_same_thread=False)
checkpointer = SqliteSaver(conn)

persistent_store = get_memory_store()
store = persistent_store.store

app = builder.compile(checkpointer=checkpointer, store=store)

In [8]:
# Example usage
config = {
    "configurable": {
        "thread_id": "user-session-1",
        "user_id": "bace",
    }
}

# Start a conversation
user_input = "Create a Python function that calculates the factorial of a number."
input_message = HumanMessage(content=user_input)

# Invoke the graph
for event in app.stream({"messages": [input_message], "user_id": "bace"}, config, stream_mode="values"):
    # Print only the last message
    if event["messages"]:
        print(event["messages"][-1].content)

Create a Python function that calculates the factorial of a number.
[DEBUG] retrieve_memories: found 0 memories for user bace


/home/bc/projects/autonomous_coding_tool/.venv/lib/python3.12/site-packages/langchain_core/language_models/base.py:354: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


[DEBUG] agent received 3 messages (after trimming)
I'll create a Python function that calculates the factorial of a number. Here's the implementation:

```python
def factorial(n):
    """
    Calculate the factorial of a non-negative integer.
    
    Args:
        n (int): A non-negative integer
        
    Returns:
        int: The factorial of n
        
    Raises:
        ValueError: If n is negative
        TypeError: If n is not an integer
    """
    if not isinstance(n, int):
        raise TypeError("Input must be an integer")
    if n < 0:
        raise ValueError("Factorial is not defined for negative numbers")
    if n == 0 or n == 1:
        return 1
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result
```

This function:
- Validates that the input is an integer
- Handles negative numbers by raising an error
- Returns 1 for 0 and 1 (base cases)
- Uses an iterative approach for efficiency

You can save this to a file using the write_file tool.

In [9]:
# Follow-up request that refers to the previous conversation
follow_up = "Now modify that function to use recursion instead of a loop."

input_message = HumanMessage(content=follow_up)

for event in app.stream(
        {"messages": [input_message], "user_id": "bace"},
        config,  # same config – thread_id = "user-session-1"
        stream_mode="values"
):
    if event["messages"]:
        print(event["messages"][-1].content)

Now modify that function to use recursion instead of a loop.
[DEBUG] retrieve_memories: found 0 memories for user bace
[DEBUG] agent received 5 messages (after trimming)
I'll modify the factorial function to use recursion instead of a loop:

```python
def factorial(n):
    """
    Calculate the factorial of a non-negative integer using recursion.
    
    Args:
        n (int): A non-negative integer
        
    Returns:
        int: The factorial of n
        
    Raises:
        ValueError: If n is negative
        TypeError: If n is not an integer
    """
    if not isinstance(n, int):
        raise TypeError("Input must be an integer")
    if n < 0:
        raise ValueError("Factorial is not defined for negative numbers")
    if n == 0 or n == 1:
        return 1
    return n * factorial(n - 1)
```

This recursive version:
- Uses the mathematical definition: n! = n × (n-1)!
- Has the same validation as the iterative version
- Handles base cases (0 and 1) correctly
- Uses memoizati

In [10]:
# First thread: teach a preference
config1 = {"configurable": {"thread_id": "thread-1", "user_id": "bace"}}
app.invoke(
    {"messages": [HumanMessage(content="Always use type hints in Python functions.")], "user_id": "bace"},
    config1
)

# Second thread: ask for code (should use type hints)
config2 = {"configurable": {"thread_id": "thread-2", "user_id": "bace"}}
response = app.invoke(
    {"messages": [HumanMessage(content="Write a function to add two numbers.")], "user_id": "bace"},
    config2
)


[DEBUG] retrieve_memories: found 0 memories for user bace
[DEBUG] agent received 2 messages (after trimming)
[Memory] Stored: User prefers type hints in Python functions.
[DEBUG] retrieve_memories: found 1 memories for user bace
[DEBUG] agent received 2 messages (after trimming)
[Memory] No relevant preference detected, skipping store.


In [11]:
response

{'messages': [HumanMessage(content='Write a function to add two numbers.', additional_kwargs={}, response_metadata={}, id='5cda3886-02f6-4a64-a031-41e2524d9594'),
  SystemMessage(content='[User preferences/patterns]\nUser prefers type hints in Python functions.\n\nUse this context when responding. For example, if the user prefers type hints, always include them.', additional_kwargs={}, response_metadata={}, id='522b8e6d-4983-45bb-8680-98c248e1e49c'),
  AIMessage(content="```python\ndef add(a, b):\n    return a + b\n```\n\nI've saved the function to a file. You can use it like this:\n\n```python\nresult = add(5, 3)\nprint(result)  # Output: 8\n```", additional_kwargs={}, response_metadata={'model': 'qwen3.5:2b', 'created_at': '2026-04-05T09:30:00.63798852Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3915374959, 'load_duration': 106882275, 'prompt_eval_count': 123, 'prompt_eval_duration': 338915089, 'eval_count': 103, 'eval_duration': 3432603062, 'logprobs': None, 'model_nam

In [15]:
# Write to a file
follow_up = "Now save the method to a file named adder.py Use write_file tool"

input_message = HumanMessage(content=follow_up)

for event in app.stream(
        {"messages": [input_message], "user_id": "bace"},
        config,  # same config – thread_id = "user-session-1"
        stream_mode="values"
):
    if event["messages"]:
        print(event["messages"][-1].content)

Now save the method to a file named adder.py Use write_file tool
[DEBUG] retrieve_memories: found 1 memories for user bace
[User preferences/patterns]
User prefers type hints in Python functions.

Use this context when responding. For example, if the user prefers type hints, always include them.
[DEBUG] agent received 10 messages (after trimming)
I'll save the factorial function to a file named adder.py using the write_file tool:

```python
def factorial(n):
    """
    Calculate the factorial of a non-negative integer using recursion.
    
    Args:
        n (int): A non-negative integer
        
    Returns:
        int: The factorial of n
        
    Raises:
        ValueError: If n is negative
        TypeError: If n is not an integer
    """
    if not isinstance(n, int):
        raise TypeError("Input must be an integer")
    if n < 0:
        raise ValueError("Factorial is not defined for negative numbers")
    if n == 0 or n == 1:
        return 1
    return n * factorial(n -